In [ ]:
import sys

sys.executable

# Módulo 2 · Expresiones regulares en Python

En esta clase vamos a aprender a **buscar, extraer, validar y transformar texto** con el módulo estándar `re`. Trabajaremos en forma incremental: literales, clases de caracteres, cuantificadores, anclas y grupos; después aplicaremos esos patrones a correos, teléfonos, fechas, registros y columnas de `pandas`.

Al terminar vas a poder elegir entre `search`, `match`, `fullmatch`, `findall`, `finditer`, `sub` y `split`, construir patrones legibles y reconocer cuándo una regex no alcanza para validar el dato.

> Todo el material de esta notebook está integrado aquí. No hace falta descargar archivos de datos. El quiz final usa los archivos del proyecto y es opcional.

## 1. Cómo leer un patrón

Una **expresión regular** es un patrón que describe texto. Se escribe como una cadena de Python; usamos cadenas crudas (`r"..."`) para que Python deje los escapes `\d`, `\s` y `\b` a cargo del motor regex.

<center>

| Patrón | Significado |
|:---:|:---|
| `gato` | Los cuatro caracteres literales `gato` |
| `.` | Cualquier carácter salvo salto de línea, por defecto |
| `\d` \| `\D` | Un dígito \| un carácter que no sea dígito |
| `\w` \| `\W` | Carácter de palabra \| carácter que no lo sea |
| `\s` \| `\S` | Espacio en blanco \| carácter que no sea espacio |
| `[aeiou]` | Una vocal minúscula de esa lista |
| `[0-9]`, `[A-Z]` | Un carácter dentro del rango |
| `[^0-9]` | Un carácter que no sea dígito; `^` niega dentro de `[]` si va primero |
| `*`, `+`, `?` | Cero o más, uno o más, cero o una repeticiones del elemento anterior |
| `{3}`, `{2,5}`, `{2,}` | Exactamente 3, entre 2 y 5, al menos 2 repeticiones |
| `^`, `$`, `\b` | Inicio, final y límite de palabra |
| `(abc)` | Grupo de captura: encuentra `abc` y guarda esa parte como grupo 1 |
| `a\|b` | Alternativa: coincide con `a` o con `b` |

</center>

Los paréntesis también sirven para agrupar alternativas. Por ejemplo, `r"(gato|perro)"` encuentra `gato` o `perro`; `group(1)` devuelve cuál de las dos palabras encontró.

En Python, `\w` y `\d` son Unicode por defecto: también pueden coincidir con letras o dígitos no ASCII. Si el formato exige dígitos ASCII, usa `[0-9]` o `re.ASCII`. Los cuantificadores suelen ser **codiciosos**: toman tanto texto como pueden; agregar `?` los vuelve perezosos, por ejemplo `.*?`.

Primero conviene leer el patrón de izquierda a derecha y probar cada pieza sobre ejemplos pequeños. `r"..."` evita que las barras invertidas se confundan con escapes de la propia cadena de Python.

## 2. Las funciones principales de `re`


- `re.search(patron, texto)` encuentra la primera coincidencia **en cualquier parte** del texto.
- `re.match(patron, texto)` solo intenta coincidir **desde el comienzo** del texto.
- `re.fullmatch(patron, texto)` exige que el patrón cubra **todo** el texto; suele ser la opción más clara para validar un formato.
- `re.findall(patron, texto)` devuelve todas las coincidencias; si hay grupos de captura, el formato de la lista depende de cuántos grupos haya.
- `re.finditer(patron, texto)` produce objetos `Match`, útiles para consultar `group()`, `span()`, `start()` y `end()`.
- `re.sub(patron, reemplazo, texto)` sustituye coincidencias; `re.subn` también devuelve el número de reemplazos.
- `re.split(patron, texto)` divide el texto donde coincide el patrón.
- `re.compile(patron, flags=0)` prepara un patrón reutilizable con métodos equivalentes.

`search` no significa “desde el inicio”: esa es la diferencia importante entre `search` y `match`. Para patrones usados varias veces, `compile` mejora la organización y evita repetir el patrón.

### Ejemplo con mails 

In [ ]:
import re

In [ ]:
help(re)

In [14]:
import re

texto = "Mi email es ana@correo.com y el soporte es soporte@tienda.com."

# Primera demostración: regex para encontrar correos de forma aproximada.

patron_email = r"[A-Za-z0-9._%-+]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}"

# Esta regex acepta, por ejemplo, ana..apellido@example.com y ana@-example.com.
# No admite todas las formas Unicode o entrecomilladas ni comprueba que el buzón exista.
emails = re.findall(patron_email, texto)
print("Emails:", emails)


Emails: ['ana@correo.com', 'soporte@tienda.com']


In [15]:

# search vs match
print("search encuentra en cualquier lugar:", bool(re.search(r"correo", texto)))
print("match exige que empiece al inicio:", bool(re.match(r"correo", texto)))
print("match con pattern que inicia:", bool(re.match(r"Mi email", texto)))

# sub: reemplazar
print(re.sub(r"\d+", "[#]", "Cuenta con 5 pedidos y 12 devoluciones"))

search encuentra en cualquier lugar: True
match exige que empiece al inicio: False
match con pattern que inicia: True
Cuenta con [#] pedidos y [#] devoluciones


In [19]:
import re

# Esto comprueba una forma habitual, no toda la especificación RFC del correo.
patron_email = re.compile(
    r"[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}"
)

for correo in [
    "ana@example.com",
    "usuario+clase@correo.org",
    "falta-arroba.example.com",
    "ana@.com",
    "ana@example.com extra",
]:
    print(f"{correo:30} -> {bool(patron_email.fullmatch(correo))}")

ana@example.com                -> True
usuario+clase@correo.org       -> True
falta-arroba.example.com       -> False
ana@.com                       -> False
ana@example.com extra          -> False


#### Opcion ya creada 

In [20]:
from email_validator import EmailNotValidError, validate_email

# Comparamos una regex didáctica con un validador especializado.
# check_deliverability=False evita consultas DNS: esto solo comprueba sintaxis.
ejemplos_correo = [
    "ana@example.com",
    "usuario+clase@correo.org",
    "ana..apellido@example.com",
    "ana@-example.com",
    "ana@example.com extra",
]

for direccion in ejemplos_correo:
    coincide_regex = patron_email.fullmatch(direccion) is not None
    try:
        resultado = validate_email(direccion, check_deliverability=False)
        acepta_validador = True
        correo_normalizado = resultado.normalized
    except EmailNotValidError:
        acepta_validador = False
        correo_normalizado = "-"

    print(
        f"{direccion:34} "
        f"regex={coincide_regex!s:5} "
        f"validador={acepta_validador!s:5} "
        f"normalizado={correo_normalizado}"
    )

ana@example.com                    regex=True  validador=True  normalizado=ana@example.com
usuario+clase@correo.org           regex=True  validador=True  normalizado=usuario+clase@correo.org
ana..apellido@example.com          regex=True  validador=False normalizado=-
ana@-example.com                   regex=True  validador=False normalizado=-
ana@example.com extra              regex=False validador=False normalizado=-


#### Nuestra solucion con regex

In [21]:
import re

patron_correo = re.compile(
    r"""
    (?=.{1,64}@)                                         # La parte anterior a @ tiene 1-64 caracteres
    [A-Za-z0-9_%+-]+                                     # Primer segmento de la parte local
    (?:\.[A-Za-z0-9_%+-]+)*                              # Puntos permitidos solo entre segmentos un solo punto
    @                                                    # un arroba literal
    [A-Za-z0-9]                                          # El dominio empieza con letra o número
    (?:[A-Za-z0-9-]{0,61}[A-Za-z0-9])?                   # Guiones solo en medio de cada etiqueta
    (?:\.[A-Za-z0-9](?:[A-Za-z0-9-]{0,61}[A-Za-z0-9])?)* # Etiquetas de dominio separadas por puntos
    \.[A-Za-z]{2,63}                                     # Un punto seguido de la extensión final min 2, max 63 caracteres
    """,
    re.VERBOSE,
)

ejemplos = [
    "ana@example.com.ar",
    "a.b@example.com",
    "ana..apellido@example.com",
    "ana@-example.com",
    "ana@example-.com",
]

for correo in ejemplos:
    print(correo, bool(patron_correo.fullmatch(correo)))

ana@example.com.ar True
a.b@example.com True
ana..apellido@example.com False
ana@-example.com False
ana@example-.com False


### Cómo leer el patrón estricto

Antes de ejecutar el patrón siguiente, fíjate en dos construcciones:

- `(?=...)` es un **lookahead positivo**: comprueba lo que viene a continuación sin consumir esos caracteres. Aquí `(?=.{1,64}@)` exige que haya entre 1 y 64 caracteres antes de `@`; no forma parte de la coincidencia.
- `(?:...)` es un **grupo sin captura**: agrupa una parte para poder cuantificarla, pero no guarda un grupo adicional. En `(?:\.[A-Za-z0-9_%+-]+)*`, la unidad repetida es “un punto y un segmento no vacío”. El `*` permite repetir esa unidad cero o más veces, por eso acepta `a.b` pero no `a..b`.

#### ¿De dónde salen 61 y 63?

DNS permite hasta **63 octetos por etiqueta** del dominio (cada tramo entre puntos). En el patrón de una etiqueta:

```text
[A-Za-z0-9](?:[A-Za-z0-9-]{0,61}[A-Za-z0-9])?
```

- El primer `[A-Za-z0-9]` reserva 1 carácter y prohíbe que la etiqueta empiece con guion.
- El último `[A-Za-z0-9]` reserva otro y prohíbe que termine con guion.
- Quedan `63 - 1 - 1 = 61` posiciones interiores; por eso el cuantificador es `{0,61}`. El `0` permite que no haya caracteres interiores.
- El grupo final es opcional (`?`), así también se permite una etiqueta de un solo carácter. Si aparece, mide de 2 a 63: `1 inicial + 0..61 interiores + 1 final`.

La extensión final usa `{2,63}`: el mínimo de 2 letras es una simplificación de este ejemplo; el máximo 63 sigue el límite de una etiqueta DNS. Como este patrón solo admite ASCII, cada carácter ocupa un octeto. Los dominios Unicode necesitarían tratamiento IDNA.

`re.VERBOSE` permite repartir la expresión en varias líneas y añadir comentarios. Después, `fullmatch()` exige que toda la dirección, y no solo un fragmento, cumpla el patrón.

Como microejemplos: `foo(?=bar)` encuentra `foo` dentro de `foobar`, sin incluir `bar`; `(?:ha)+` agrupa y repite `ha` sin crear una captura.

## 3. Patrones prácticos: el formato y sus límites

| Dato | Patrón de forma | Nota |
|---|---|---|
| Teléfono `555-123-4567` | `r"\d{3}-\d{3}-\d{4}"` | Comprueba el formato, no si el número existe |
| Fecha ISO `2026-09-27` | `r"\d{4}-\d{2}-\d{2}"` | Comprueba la forma; `datetime` verifica el calendario |
| Palabra completa | `r"\bPython\b"` | Evita encontrar `Python` dentro de `MicroPython` |
| Correo común | `r"[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}"` | Aproximación práctica, no validador RFC completo |
| URL simple | `r"https?://[^\s]+"` | Puede incluir puntuación final del texto: recórtala después |

Para **buscar** en un texto se puede dejar el patrón sin anclas. Para validar que una cadena completa respeta un formato, preferí `re.fullmatch`; no aceptes una coincidencia parcial por accidente.

In [23]:
import re

registros = """
Ana López <ana.lopez@example.com> llamó el 2026-09-27 desde 11-4828-7186.
El pedido B-204 se envió a https://tienda.example/pedidos/204.
Soporte: soporte+ventas@ejemplo.org; teléfono alternativo 555-123-4567.
"""

patrones = {
    "correos": r"[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}",
    "teléfonos": r"\b\d{3}-\d{3}-\d{4}\b",
    "fechas ISO": r"\b\d{4}-\d{2}-\d{2}\b",
    "URLs": r"https?://[^\s]+",
}

for etiqueta, patron in patrones.items():
    coincidencias = re.findall(patron, registros)
    if etiqueta == "URLs":
        coincidencias = [url.rstrip(".,;:!?)") for url in coincidencias]
    print(f"{etiqueta}: {coincidencias}")

correos: ['ana.lopez@example.com', 'soporte+ventas@ejemplo.org']
teléfonos: ['555-123-4567']
fechas ISO: ['2026-09-27']
URLs: ['https://tienda.example/pedidos/204']


## 4. Grupos: capturar, nombrar y recuperar partes

Los paréntesis agrupan y, por defecto, **capturan**. `group(0)` es la coincidencia completa; `group(1)`, `group(2)` y `groups()` devuelven las capturas. Los grupos con nombre (`(?P<nombre>...)`) hacen el código más legible. Si solo necesitás agrupar para aplicar `|` o un cuantificador, usa `(?:...)` y no crees una captura adicional.

In [24]:
import re

texto = "Contacto: 555-123-4567; alternativo: (212) 555-0198"
patron = re.compile(
    r"(?P<area>\d{3})-(?P<prefijo>\d{3})-(?P<numero>\d{4})"
)

for coincidencia in patron.finditer(texto):
    print("completo:", coincidencia.group(0))
    print("partes:", coincidencia.groupdict())
    print("posición:", coincidencia.span())

completo: 555-123-4567
partes: {'area': '555', 'prefijo': '123', 'numero': '4567'}
posición: (10, 22)


In [25]:
coincidencia

<re.Match object; span=(10, 22), match='555-123-4567'>

In [28]:
coincidencia.group(4)

IndexError: no such group

## 5. Regex con `pandas`

Las cadenas de una columna se pueden procesar con `Series.str`: por ejemplo, `.contains(patron, regex=True)` filtra filas, `.extract(patron)` extrae grupos de captura, `.replace(patron, reemplazo, regex=True)` limpia texto y `.fullmatch(patron)` valida la cadena completa. `na=False` hace explícito cómo tratar valores ausentes.

En las siguientes celdas creamos datos dentro de la notebook, extraemos un correo y una fecha, y consultamos qué registros tienen un teléfono con el formato esperado.

In [29]:
import pandas as pd

contactos = pd.DataFrame({
    "nombre": ["Ana", "Luis", "Mora"],
    "detalle": [
        "ana.lopez@example.com | 555-123-4567",
        "luis+ventas@correo.org | sin teléfono",
        "correo pendiente | 212-555-0198",
    ],
    "fecha": ["2026-09-27", "27/09/2026", "2026-10-03"],
})
contactos

,nombre,detalle,fecha
0,Ana,ana.lopez@example.com | 555-123-4567,2026-09-27
1,Luis,luis+ventas@correo.org | sin teléfono,27/09/2026
2,Mora,correo pendiente | 212-555-0198,2026-10-03


In [30]:
patron_correo = r"([A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,})"
patron_telefono = r"(\b\d{3}-\d{3}-\d{4}\b)"

contactos["correo"] = contactos["detalle"].str.extract(
    patron_correo, expand=False
)
contactos["telefono"] = contactos["detalle"].str.extract(
    patron_telefono, expand=False
)
contactos["fecha_con_guiones"] = contactos["fecha"].str.replace(
    r"/", "-", regex=True
)
contactos

,nombre,detalle,fecha,correo,telefono,fecha_con_guiones
0,Ana,ana.lopez@example.com | 555-123-4567,2026-09-27,ana.lopez@example.com,555-123-4567,2026-09-27
1,Luis,luis+ventas@correo.org | sin teléfono,27/09/2026,luis+ventas@correo.org,NaN,27-09-2026
2,Mora,correo pendiente | 212-555-0198,2026-10-03,NaN,212-555-0198,2026-10-03


In [31]:
patron_telefono_busqueda = r"\b\d{3}-\d{3}-\d{4}\b"
mascara = contactos["detalle"].str.contains(
    patron_telefono_busqueda, regex=True, na=False
)
print("Registros con teléfono:")
display(contactos.loc[mascara, ["nombre", "telefono"]])

print("Cantidad de correos extraídos:", contactos["correo"].notna().sum())

Registros con teléfono:


,nombre,telefono
0,Ana,555-123-4567
2,Mora,212-555-0198


Cantidad de correos extraídos: 2


---
# Quiz interactivo: Regex y limpieza de datos

Respondé cada pregunta y pulsá **Ver respuesta**. Hay preguntas de una sola opción y de múltiple respuesta.


In [2]:
import json
import sys
from pathlib import Path

bases = (Path.cwd(), *Path.cwd().parents)
RAIZ = next((base for base in bases if (base / "quiz").is_dir()), None)
if RAIZ is None:
    raise FileNotFoundError("No se encontró la carpeta quiz del proyecto")
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))

from quiz.quiz import aplicar_estilos, crear_quiz, crear_quiz_multiple

candidatas = [Path.cwd() / "Modulo_2_Regex_preguntas.json"]
candidatas.extend(
    base / "Automatizacion" / "Modulo_2" / "Modulo_2_Regex_preguntas.json"
    for base in bases
)
RUTA_PREGUNTAS = next((ruta for ruta in candidatas if ruta.is_file()), None)
if RUTA_PREGUNTAS is None:
    raise FileNotFoundError("No se encontró Modulo_2_Regex_preguntas.json")

with RUTA_PREGUNTAS.open(encoding="utf-8") as archivo:
    respuestas = json.load(archivo)

aplicar_estilos()
print(f"Quiz listo: {len(respuestas)} preguntas")

Quiz listo: 10 preguntas


### Pregunta 1

Necesitás extraer **direcciones de correo** de grandes bloques de texto. ¿Qué expresión regular es la más adecuada?

In [3]:
q1_data = respuestas["q1"]
crear_quiz(q1_data["opciones"], q1_data["correcta"], q1_data["explicacion"])

RadioButtons(layout=Layout(width='100%'), options=(('A) \\d+', 'A'), ('B) [A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\\.[…

Button(description='Ver respuesta', layout=Layout(width='auto'), style=ButtonStyle())

Output()

### Pregunta 2

Qué significa el cuantificador `+` en un patrón regex (p. ej. en `\d+`)?

In [4]:
q2_data = respuestas["q2"]
crear_quiz(q2_data["opciones"], q2_data["correcta"], q2_data["explicacion"])

RadioButtons(layout=Layout(width='100%'), options=(('A) Cualquier dígito', 'A'), ('B) Uno o más dígitos', 'B')…

Button(description='Ver respuesta', layout=Layout(width='auto'), style=ButtonStyle())

Output()

### Pregunta 3

Qué tipo de datos coincide el patrón `\d{3}-\d{3}-\d{4}`?

In [5]:
q3_data = respuestas["q3"]
crear_quiz(q3_data["opciones"], q3_data["correcta"], q3_data["explicacion"])

RadioButtons(layout=Layout(width='100%'), options=(('A) Números de teléfono con formato XXX-XXX-XXXX', 'A'), (…

Button(description='Ver respuesta', layout=Layout(width='auto'), style=ButtonStyle())

Output()

### Pregunta 4

Cuáles de las siguientes **funciones** pertenecen al módulo `re`? (Seleccioná todas las correctas)

In [6]:
q4_data = respuestas["q4"]
crear_quiz_multiple(q4_data["opciones"], q4_data["correctas"], q4_data["explicacion"])

Checkbox(value=False, description='A) re.search() para encontrar el primer patrón', indent=False, layout=Layou…

Checkbox(value=False, description='B) re.match() para anclar el patrón al inicio', indent=False, layout=Layout…

Checkbox(value=False, description='C) re.findall() para devolver todas las coincidencias', indent=False, layou…

Checkbox(value=False, description='D) re.sub() para reemplazar coincidencias', indent=False, layout=Layout(wid…

Button(description='Ver respuesta', layout=Layout(width='auto'), style=ButtonStyle())

Output()

### Pregunta 5

Necesitás obtener **todas** las coincidencias de un patrón en un texto. ¿Qué función usás?

In [7]:
q5_data = respuestas["q5"]
crear_quiz(q5_data["opciones"], q5_data["correcta"], q5_data["explicacion"])

RadioButtons(layout=Layout(width='100%'), options=(('A) re.search()', 'A'), ('B) re.match()', 'B'), ('C) re.fi…

Button(description='Ver respuesta', layout=Layout(width='auto'), style=ButtonStyle())

Output()

### Pregunta 6

¿Qué ancla de regex exige que la coincidencia ocurra **al inicio** de la línea?

In [8]:
q6_data = respuestas["q6"]
crear_quiz(q6_data["opciones"], q6_data["correcta"], q6_data["explicacion"])

RadioButtons(layout=Layout(width='100%'), options=(('A) ^', 'A'), ('B) $', 'B'), ('C) \\b', 'C'), ('D) .', 'D'…

Button(description='Ver respuesta', layout=Layout(width='auto'), style=ButtonStyle())

Output()

### Pregunta 7

Para validar un número de teléfono con el formato `XXX-XXX-XXXX`, qué patrón usarías?

In [9]:
q7_data = respuestas["q7"]
crear_quiz(q7_data["opciones"], q7_data["correcta"], q7_data["explicacion"])

RadioButtons(layout=Layout(width='100%'), options=(('A) r"\\d+-\\d+-\\d+"', 'A'), ('B) r"\\w{3}-\\w{3}-\\w{4}"…

Button(description='Ver respuesta', layout=Layout(width='auto'), style=ButtonStyle())

Output()

### Pregunta 8

¿Qué clase de caracteres coincide con **cualquier dígito** (0-9)?

In [10]:
q8_data = respuestas["q8"]
crear_quiz(q8_data["opciones"], q8_data["correcta"], q8_data["explicacion"])

RadioButtons(layout=Layout(width='100%'), options=(('A) \\w', 'A'), ('B) \\d', 'B'), ('C) \\s', 'C'), ('D) \\b…

Button(description='Ver respuesta', layout=Layout(width='auto'), style=ButtonStyle())

Output()

### Pregunta 9

Querés extraer todas las fechas en formato `AAAA-MM-DD` de archivos de registro. ¿Qué patrones funcionan? (Seleccioná todas las correctas)

In [11]:
q9_data = respuestas["q9"]
crear_quiz_multiple(q9_data["opciones"], q9_data["correctas"], q9_data["explicacion"])

Checkbox(value=False, description='A) \\d{4}-\\d{2}-\\d{2}', indent=False, layout=Layout(width='100%'))

Checkbox(value=False, description='B) [0-9]{4}-[0-9]{2}-[0-9]{2}', indent=False, layout=Layout(width='100%'))

Checkbox(value=False, description='C) \\d{4}/\\d{2}/\\d{2}', indent=False, layout=Layout(width='100%'))

Checkbox(value=False, description='D) [A-Za-z]+ \\d{1,2}, \\d{4}', indent=False, layout=Layout(width='100%'))

Button(description='Ver respuesta', layout=Layout(width='auto'), style=ButtonStyle())

Output()

### Pregunta 10

Tenés un CSV con duplicados, valores faltantes y tipos mezclados. ¿Qué biblioteca es la más útil para limpiarlo y transformarlo?

In [12]:
q10_data = respuestas["q10"]
crear_quiz(q10_data["opciones"], q10_data["correcta"], q10_data["explicacion"])

RadioButtons(layout=Layout(width='100%'), options=(('A) matplotlib', 'A'), ('B) pandas', 'B'), ('C) argparse',…

Button(description='Ver respuesta', layout=Layout(width='auto'), style=ButtonStyle())

Output()

## Fin del quiz y transición

El quiz repasa los conceptos básicos. Debajo continúa la guía práctica con cuantificadores, anclas, grupos nombrados, banderas, limpieza y validación. Para repasar la clase de forma independiente, volvé a ejecutar las celdas en orden.

# Guía de profundización

El quiz sirve para comprobar los conceptos básicos. Las secciones siguientes conectan cada pieza de sintaxis con una salida ejecutable. Ejecuta las celdas en orden y cambia los ejemplos para ver cómo responde el patrón.

## 6. Clases, cuantificadores y límites de palabra

Una regex se construye en piezas: clase de caracteres, cantidad de repeticiones y, si hace falta, una posición. Dentro de una clase `[^...]`, `^` niega el conjunto; fuera de corchetes, `^` ancla al inicio. `\b` marca un límite de palabra, no un espacio. Los cuantificadores son codiciosos por defecto: `.*` toma lo máximo posible; `.*?` intenta tomar lo mínimo.

En Python `\d` y `\w` reconocen caracteres Unicode. Si la regla exige solo dígitos ASCII, usa `[0-9]`. Prueba los casos límite, no solo el ejemplo que debería coincidir.

In [16]:
import re

texto = "A A1 A12 A123 abc abbc abbbc a123"
print("dos o tres cifras tras A:", re.findall(r"A\d{2,3}", texto))
print("una o más b:", re.findall(r"ab+c", "ac abc abbc abbbc"))


dos o tres cifras tras A: ['A12', 'A123']
una o más b: ['abc', 'abbc', 'abbbc']


In [22]:

html = "<uno> <hols que tal> <dos>"
print("codicioso:", re.findall(r"<.*>", html))
print("perezoso:", re.findall(r"<.*?>", html))


codicioso: ['<uno> <hols que tal> <dos>']
perezoso: ['<uno>', '<hols que tal>', '<dos>']


In [ ]:
palabras = "Python MicroPython regex"
print("sin límite:", re.findall(r"Python", palabras))
print("palabra completa:", re.findall(r"\bPython\b", palabras))

sin límite: ['Python', 'Python']
palabra completa: ['Python']


In [27]:
palabras = "MicroPythonregex"
print("sin límite:", re.search(r"Python", palabras))
print("palabra completa:", re.search(r"\bPython\b", palabras))

sin límite: <re.Match object; span=(5, 11), match='Python'>
palabra completa: None


In [28]:

palabras = "Python MicroPython regex"
print("sin límite:", re.findall(r"Python", palabras))
print("palabra completa:", re.findall(r"\bPython\b", palabras))

sin límite: ['Python', 'Python']
palabra completa: ['Python']


## 7. Anclas, líneas y las tres formas de buscar

`^` y `$` anclan el principio y el final; con `re.MULTILINE` lo hacen para cada línea. `\A` y `\Z` se refieren a los extremos del texto completo. Para encontrar en cualquier posición usa `search`; para exigir un prefijo, `match`; para validar la cadena entera, `fullmatch`.

Un `Match` guarda la cadena, las posiciones y las capturas. Usa `group(0)` para la coincidencia completa, `span()` para su rango y grupos con nombre para que la extracción se explique sola.

In [29]:
import re

texto = "INFO inicio\nERROR disco lleno\nfin"
print("texto:")
print(texto)
patron_linea = r"^ERROR:?.*$"
print("error en cualquier línea:", re.search(patron_linea, texto, re.MULTILINE).group())


texto:
INFO inicio
ERROR disco lleno
fin
error en cualquier línea: ERROR disco lleno


In [49]:
texto = """"
INFO inicio
ERROR: disco lleno´
ERROR disco lleno´
INFO proceso intermedio
ERROR
"""
print("texto:")
print(texto)
patron_linea = r"^ERROR:?.*?$"
print("error en cualquier línea:", re.findall(patron_linea, texto, re.MULTILINE))

texto:
"
INFO inicio
ERROR: disco lleno´
ERROR disco lleno´
INFO proceso intermedio
ERROR

error en cualquier línea: ['ERROR: disco lleno´', 'ERROR disco lleno´', 'ERROR']


In [50]:

codigo = re.compile(r"(?P<area>[A-Z]{2})-(?P<numero>\d{4})")
texto_codigos = "Pedido AB-2048; otro ZZ-0017"
for coincidencia in codigo.finditer(texto_codigos):
    print(coincidencia.group(0), coincidencia.groupdict(), coincidencia.span())

print("match al inicio:", re.match(codigo, texto_codigos))
print("fullmatch completo:", bool(codigo.fullmatch("AB-2048")))

AB-2048 {'area': 'AB', 'numero': '2048'} (7, 14)
ZZ-0017 {'area': 'ZZ', 'numero': '0017'} (21, 28)
match al inicio: None
fullmatch completo: True


## 8. Banderas para patrones más legibles

Las banderas cambian el comportamiento sin volver ilegible el patrón: `re.IGNORECASE` ignora mayúsculas, `re.MULTILINE` ajusta `^`/`$` a cada línea, `re.DOTALL` permite que `.` cruce saltos de línea y `re.VERBOSE` permite escribir el patrón en varias líneas con espacios y comentarios. En Python, Unicode es el comportamiento predeterminado; `re.ASCII` restringe clases abreviadas como `\w` y `\d`.

In [51]:
import re

texto = "inicio\nERROR: disco lleno\nfin"
print(re.findall(r"^ERROR:.*$", texto, flags=re.MULTILINE))

patron_codigo = re.compile(
    r"""
    (?P<grupo> [A-Z]{2} )  # dos letras mayúsculas
    -
    (?P<numero> \d{4} )   # cuatro dígitos
    """,
    flags=re.VERBOSE,
)
print(patron_codigo.fullmatch("AB-2048").groupdict())

['ERROR: disco lleno']
{'grupo': 'AB', 'numero': '2048'}


## 9. Sustituir, separar y escapar texto literal

`re.sub()` reemplaza cada coincidencia; `re.subn()` también devuelve cuántas reemplazó. Las capturas nombradas pueden reutilizarse en el reemplazo con `\g<nombre>`. `re.split()` divide por un patrón. Si el patrón debe contener texto literal ingresado por alguien, pásalo primero por `re.escape()` para que puntos, `+` o paréntesis no se interpreten como metacaracteres.

In [52]:
import re

fecha = "27/09/2026"
patron_fecha = r"(?P<dia>\d{2})/(?P<mes>\d{2})/(?P<anio>\d{4})"
print(re.sub(patron_fecha, r"\g<anio>-\g<mes>-\g<dia>", fecha))


2026-09-27


In [54]:

texto = "ana@example.com; luis@example.org, mora@example.net"
print(re.split(r"\s*[,;.]\s*", texto))


['ana@example', 'com', 'luis@example', 'org', 'mora@example', 'net']


In [57]:

literal = "precio (final) + IVA"
patron_literal = re.compile(re.escape(literal), re.IGNORECASE)
print(patron_literal.pattern)
print(bool(patron_literal.search("Se calcula el PRECIO (FINAL) + IVA.")))

precio\ \(final\)\ \+\ IVA
True


## 10. Validar el formato no es validar el dato

Usa `fullmatch()` cuando la cadena completa deba seguir un formato. Regex puede comprobar que una fecha tenga forma `AAAA-MM-DD`, pero no que ese día exista; para eso usa `date.fromisoformat()`. Un patrón sencillo de correo sirve para filtros y ejercicios, pero no cubre toda la especificación ni confirma que el buzón exista. La validación debe corresponderse con el requisito real.

In [58]:
import re
from datetime import date

patron_fecha_iso = re.compile(r"\d{4}-\d{2}-\d{2}")

def es_fecha_iso(valor):
    if not patron_fecha_iso.fullmatch(valor):
        return False
    try:
        date.fromisoformat(valor)
    except ValueError:
        return False
    return True

for valor in ["2024-02-29", "2025-02-29", "2026-13-01"]:
    print(valor, "->", es_fecha_iso(valor))

patron_usuario = re.compile(r"[A-Za-z_][A-Za-z_0-9]*")
print("_dato7 ->", bool(patron_usuario.fullmatch("_dato7")))
print("7dato  ->", bool(patron_usuario.fullmatch("7dato")))

2024-02-29 -> True
2025-02-29 -> False
2026-13-01 -> False
_dato7 -> True
7dato  -> False


## 11. Aplicación: extraer campos de un log

Los grupos con nombre son especialmente útiles cuando el texto sigue un formato repetido. Con `finditer()` obtenemos cada registro como `Match`; con `groupdict()` convertimos las capturas en un diccionario con nombres claros. La bandera `MULTILINE` hace que `^` vuelva a coincidir al inicio de cada línea.

In [59]:
import re

logs = """2026-09-27 INFO servicio iniciado
2026-09-27 ERROR no se pudo abrir el archivo
2026-09-28 WARN reintento programado"""

patron_log = re.compile(
    r"(?P<fecha>\d{4}-\d{2}-\d{2}) "
    r"(?P<nivel>INFO|WARN|ERROR) (?P<mensaje>.*)$",
    flags=re.MULTILINE,
)

for registro in patron_log.finditer(logs):
    print(registro.groupdict())

{'fecha': '2026-09-27', 'nivel': 'INFO', 'mensaje': 'servicio iniciado'}
{'fecha': '2026-09-27', 'nivel': 'ERROR', 'mensaje': 'no se pudo abrir el archivo'}
{'fecha': '2026-09-28', 'nivel': 'WARN', 'mensaje': 'reintento programado'}


## 12. Práctica para intentar

Antes de ver las soluciones, escribe un patrón y pruébalo tanto con un caso que debe coincidir como con uno que debe rechazarse.

1. Extrae códigos como `AB-2048`: dos mayúsculas, guion y cuatro dígitos, como palabras completas.
2. Valida que toda una cadena sea un identificador que empieza con letra o `_` y continúa con letras, dígitos o `_`.
3. Colapsa espacios, tabulaciones y saltos de línea repetidos en un único espacio y recorta los extremos.
4. Extrae día, mes y año de `27/09/2026` con grupos nombrados.
5. Detecta palabras repetidas de manera consecutiva.

Las soluciones están inmediatamente debajo; intenta cada una primero en una celda propia.

In [60]:
import re

# 1. Extraer códigos completos.
print(re.findall(r"\b[A-Z]{2}-\d{4}\b", "AB-2048 y ZZ-0017"))

# 2. Validar el identificador completo.
patron_usuario = re.compile(r"[A-Za-z_][A-Za-z_0-9]*")
print(bool(patron_usuario.fullmatch("_dato7")))
print(bool(patron_usuario.fullmatch("7dato")))

# 3. Normalizar espacios.
print(re.sub(r"\s+", " ", "  hola\t\nmundo  ").strip())

# 4. Capturar partes de una fecha.
fecha = re.fullmatch(
    r"(?P<dia>\d{2})/(?P<mes>\d{2})/(?P<anio>\d{4})",
    "27/09/2026",
)
print(fecha.groupdict())

# 5. Detectar una palabra repetida.
patron_repetida = r"\b(?P<palabra>\w+)\s+(?P=palabra)\b"
print(re.findall(patron_repetida, "muy muy bien"))

['AB-2048', 'ZZ-0017']
True
False
hola mundo
{'dia': '27', 'mes': '09', 'anio': '2026'}
['muy']


## Resumen rápido

| Si necesitas... | Usa... |
|---|---|
| Buscar la primera coincidencia en cualquier parte | `re.search()` |
| Exigir coincidencia al comienzo | `re.match()` |
| Validar la cadena completa | `re.fullmatch()` |
| Obtener todas las coincidencias | `re.findall()` o `re.finditer()` |
| Ver posiciones y grupos | `Match.span()`, `Match.groupdict()` |
| Sustituir, dividir o escapar texto literal | `re.sub()`, `re.split()`, `re.escape()` |
| Hacer legible un patrón largo | `re.compile(..., re.VERBOSE)` |
| Validar una regla semántica (por ejemplo, calendario) | Regex para la forma y una función especializada para el significado |

**Regla de oficio:** empieza con el patrón más simple que describa el requisito, nómbralo si se reutiliza y pruébalo con ejemplos válidos, inválidos y de borde.

# Mas Ejemplos

## A. Números decimales y subgrupos

En este ejemplo aparecen decimales dentro de una oración. `match` solo los encuentra si el texto empieza por el número; `search` localiza el primero en cualquier posición y `finditer` permite recorrerlos y consultar cada grupo.

In [ ]:
import re

texto = "a 3.14 hola aca voy 12.56"
patron_decimal = re.compile(r"(?P<entero>\d+)\.(?P<decimal>\d+)")

print("match desde el inicio:", re.match(patron_decimal, texto))
print("primer decimal:", re.search(patron_decimal, texto).groupdict())

for coincidencia in patron_decimal.finditer(texto):
    print(coincidencia.group(), coincidencia.groups(), coincidencia.span())

## B. Filtrar listas de nombres y palabras

las anclas, rangos y clases como `[g-k]`, `[hxjv]`, `niñ[ao]s` y `se[0-5]`. La función siguiente conserva esa idea de filtrar una lista, pero compila el patrón una sola vez y usa `search` porque queremos encontrarlo en cualquier parte de cada elemento.

In [61]:
import re

nombres = [
    "ana gomez",
    "luca passarello",
    "mica porras",
    "belen cianfagna",
    "federico porras",
]

def filtrar(patron, valores, flags=0):
    regex = re.compile(patron, flags)
    return [valor for valor in valores if regex.search(valor)]


In [62]:

print("empiezan con be:", filtrar(r"^be", nombres, re.IGNORECASE))


empiezan con be: ['belen cianfagna']


In [63]:

print("terminan en o:", filtrar(r"o$", nombres, re.IGNORECASE))


terminan en o: ['luca passarello']


In [64]:
print("contienen letras entre g y k:", filtrar(r"[g-k]", nombres))


contienen letras entre g y k: ['ana gomez', 'mica porras', 'belen cianfagna', 'federico porras']


In [65]:

codigos = ["se1", "se9", "ma2", "se:", "se.", "se3"]
print("se seguido de un dígito de 0 a 5:", filtrar(r"se[0-5]", codigos))


se seguido de un dígito de 0 a 5: ['se1', 'se3']


In [66]:

palabras = "hombre mujer niños niñas mascota años".split()
print("contienen h, x, j o v:", filtrar(r"[hxjv]", palabras))
print("niño o niña:", filtrar(r"niñ[ao]s", palabras))

contienen h, x, j o v: ['hombre', 'mujer']
niño o niña: ['niños', 'niñas']


## C. Extraer correos y teléfonos de un mismo texto

**Origen:** `ej_re_extraerDatos.py` y `el_re19_buscar mails.py`.

La idea original combina varios formatos con alternancia (`|`). Aquí los distinguimos con grupos nombrados. También corregimos un detalle del patrón de correo original: el punto del dominio se escribe `\.` para buscar un punto literal, no cualquier carácter.

In [ ]:
import re

contacto = """Consultas: 4904-1859 / 15-6050-1859 / 11-4828-7186
Mail: graduadosfiuba@gmail.com
WhatsApp: 11 6050 1859
En Luján: 02323-234-9087"""

patron_correo = r"[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}"
patron_telefono = r"(?:\d{2,5}[\s-]\d{3,4}[\s-]\d{4}|\d{4}[\s-]\d{4})"
patron_contacto = re.compile(
    rf"(?P<correo>{patron_correo})|(?P<telefono>{patron_telefono})"
)

for coincidencia in patron_contacto.finditer(contacto):
    tipo = "correo" if coincidencia.group("correo") else "teléfono"
    print(tipo, "->", coincidencia.group())

## D. Encontrar y reordenar fechas

**Origen:** `el_re07sub.py` y `el_re08fechas.py`.

Tus scripts buscan fechas con separadores variados y cambian el orden mediante referencias a grupos (`\1`, `\2`, `\3`). Acá usamos grupos nombrados y hacemos que el separador se repita igual en ambos lugares; así `2020-09/10` no pasa por una fecha bien formada.

In [ ]:
import re

registros = "2012/01/12; 1997.04.02; 1815-07-26; 2020-09/10"
patron_fecha = re.compile(
    r"\b(?P<anio>\d{4})(?P<separador>[-/.])"
    r"(?P<mes>\d{2})(?P=separador)(?P<dia>\d{2})\b"
)
print("fechas encontradas:", [m.group() for m in patron_fecha.finditer(registros)])

fecha_dmy = "27/09/2026"
patron_dmy = r"(?P<dia>\d{2})/(?P<mes>\d{2})/(?P<anio>\d{4})"
fecha_iso = re.sub(
    patron_dmy,
    r"\g<anio>-\g<mes>-\g<dia>",
    fecha_dmy,
)
print("fecha reordenada:", fecha_iso)

## E. Espacios, caracteres de control y tildes

**Origen:** `borrar_espacios.py` y `borrar_caracteres.py`.

Tus ejercicios comparan `strip`, `lstrip`, `rstrip`, quitar todos los espacios y colapsar espacios repetidos. Son operaciones distintas. La regex `\s+` también cubre tabulaciones y saltos de línea. El script de caracteres además normaliza Unicode para quitar marcas diacríticas: lo mostramos como transformación opcional, porque quitar tildes puede cambiar palabras y `ñ` pasa a `n`.

In [69]:
import re
import unicodedata

def quitar_marcas_diacriticas(texto):
    normalizado = unicodedata.normalize("NFD", texto)
    return "".join(
        caracter
        for caracter in normalizado
        if not unicodedata.combining(caracter)
    )

texto = "  Pýtĥón\t\f es genial\r que\r bueno    es aprender  "
print("recortar extremos:", ascii(texto.strip()))
print("colapsar whitespace:", ascii(re.sub(r"\s+", " ", texto).strip()))
print("sin diacríticos:", quitar_marcas_diacriticas("Pýtĥón"))
print("advertencia: niño pasa a", quitar_marcas_diacriticas("niño"))

recortar extremos: 'P\xfdt\u0125\xf3n\t\x0c es genial\r que\r bueno    es aprender'
colapsar whitespace: 'P\xfdt\u0125\xf3n es genial que bueno es aprender'
sin diacríticos: Python
advertencia: niño pasa a nino


## F. Leer registros de jugadores con campos nombrados

**Origen:** `jugadores_river.py`.

Tu archivo reúne nombre, dos campos geográficos, posición y fecha, y el script los separa por whitespace para acceder por índice. Acá expresamos el formato con grupos nombrados: el nombre puede tener varias palabras, y cada campo queda etiquetado. La fecha se valida aparte, porque una secuencia con forma `00.00.0000` no es una fecha real.

In [ ]:
import re
from datetime import datetime

registros_jugadores = [
    "Gonzalo Abán Argentina Argentina Delantero 11.06.1987",
    "Juan_Pablo Ángel Colombia Colombia Delantero 24.10.1975",
    "Mario Artel Argentina Argentina Delantero 00.00.0000",
]

patron_jugador = re.compile(
    r"(?P<nombre>.+?)\s+(?P<campo_1>\S+)\s+(?P<campo_2>\S+)\s+"
    r"(?P<posicion>Portero|Defensa|Centrocampista|Delantero)\s+"
    r"(?P<fecha>\d{2}\.\d{2}\.\d{4})"
)

for linea in registros_jugadores:
    coincidencia = patron_jugador.fullmatch(linea)
    if coincidencia is None:
        print("formato no reconocido:", linea)
        continue
    jugador = coincidencia.groupdict()
    try:
        jugador["fecha"] = datetime.strptime(
            jugador["fecha"], "%d.%m.%Y"
        ).date().isoformat()
    except ValueError:
        jugador["fecha"] = None
    print(jugador)

## G. Convertir una expresión en tokens

**Origen:** `ej_re_scanner.py`.

Tu script asigna un grupo con nombre a cada tipo (`variable`, `número`, `suma`, `producto`, `igual`, `división`). La versión siguiente mantiene esa idea y usa `finditer`, comprobando los offsets para no saltarse silenciosamente un carácter inválido. Es un primer tokenizer didáctico, no un parser ni una calculadora.

In [ ]:
import re

tipos_token = [
    ("ESPACIO", r"\s+"),
    ("NUMERO", r"\d+(?:\.\d+)?"),
    ("VARIABLE", r"[A-Za-z_]\w*"),
    ("SUMA", r"\+"),
    ("PRODUCTO", r"\*"),
    ("IGUAL", r"="),
    ("DIVISION", r"/"),
]
patron_token = re.compile(
    "|".join(f"(?P<{nombre}>{patron})" for nombre, patron in tipos_token)
)

def tokenizar(expresion):
    tokens = []
    posicion = 0
    for coincidencia in patron_token.finditer(expresion):
        if coincidencia.start() != posicion:
            raise ValueError(f"carácter no reconocido en posición {posicion}")
        posicion = coincidencia.end()
        if coincidencia.lastgroup != "ESPACIO":
            tokens.append((coincidencia.lastgroup, coincidencia.group()))
    if posicion != len(expresion):
        raise ValueError(f"carácter no reconocido en posición {posicion}")
    return tokens

print(tokenizar("resul =13+6* 2/3"))

## H. Extraer números y decimales

**Origen:** `el_re09_buscar cantidad de caracteres.py`, `el_re10_buscar numeros..py` y `extraer_numeros.py`.

En el script original aparece `.` dentro de un patrón numérico; fuera de `[...]`, el punto significa «cualquier carácter». Para el punto decimal literal hay que escribir `\.`. Este patrón acepta enteros, decimales con punto inicial y signo opcional; los límites negativos evitan extraer pedazos de palabras o números con más puntos.

In [ ]:
import re

texto = "emiliano 45 belen 44 200.52, .235, 1.235 y otra.g"
patron_numero = re.compile(
    r"(?<![\w.])[+-]?(?:\d+(?:\.\d+)?|\.\d+)(?![\w.])"
)

coincidencias = [m.group() for m in patron_numero.finditer(texto)]
print("fragmentos numéricos:", coincidencias)
print("como float:", [float(valor) for valor in coincidencias])

# Palabras de una a seis letras, como en el ejercicio de conteo.
print(re.findall(r"\b[A-Za-z]{1,6}\b", "Optimizá el código para Python"))

## I. Comodines de archivos no son regex

**Origen:** `ej_fnmach.py`, `indexall.py` y `el_re01.py`.

`fnmatch` usa patrones de nombres de archivo como `*.txt`; no usa la misma sintaxis que `re`. Si la tarea es encontrar literalmente una subcadena repetida, `re.escape()` combinado con `finditer()` devuelve todas sus posiciones sin escribir una función de búsqueda manual.

In [ ]:
import fnmatch
import re

archivo = "producto.txt"
print("comodín fnmatch:", fnmatch.fnmatch(archivo, "*.txt"))
print("regex:", bool(re.fullmatch(r".*\.txt", archivo)))

texto = "River Plate; otro River, y River otra vez"
literal = "River"
posiciones = [
    coincidencia.start()
    for coincidencia in re.finditer(re.escape(literal), texto)
]
print("posiciones de River:", posiciones)